In [113]:
import sys
import random
import math
import logging
import json
import os

logging.basicConfig(
    filename = "game.log",
    level = logging.INFO,
    format =  "%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

class Game:

    def __init__(self):
        self.mode = None
        self.last_guessDiff = None
        self.last_matchDiff = None
        self.state = "menu"
        self.game_properties = {
            "menu": self.menu,
            "mode": self.mode_pick,
            "guess": self.guess_difficulty,
            "match": self.match_difficulty,
            "stats": self.view_stats,
            "guess_easy": lambda: self.guessing_play(1, 10, math.inf),
            "guess_medium": lambda: self.guessing_play(1, 100, 5),
            "guess_hard": lambda: self.guessing_play(1, 1000, 3),
            "match_easy": lambda: self.matching_play(1, 20, math.inf),
            "match_medium": lambda: self.matching_play(1, 100, 5),
            "match_hard": lambda: self.matching_play(1, 1000, 3),
            "continue": self.game_end
        }
        self.stats = {}
        
    def create_stats(self):
        if not os.path.exists("stats.json"):
            with open("stats.json", "w") as file:
                json.dump({
                    "games_played": 0,
                    "wins": 0,
                    "wins_with_attempts": 0,
                    "losses": 0,
                    "games_played_mode_guess": 0,
                    "wins_mode_guess": 0,
                    "losses_mode_guess": 0,
                    "games_guess_easy": 0,
                    "guess_easy_wins": 0,
                    "games_guess_medium": 0,
                    "guess_medium_wins": 0,
                    "guess_medium_wins_with_attempts": 0,
                    "guess_medium_losses": 0,
                    "games_guess_hard": 0,
                    "guess_hard_wins": 0,
                    "guess_hard_wins_with_attempts": 0,
                    "guess_hard_losses": 0,
                    "games_played_mode_match": 0,
                    "wins_mode_match": 0,
                    "losses_mode_match": 0,
                    "games_match_easy": 0,
                    "match_easy_wins": 0,
                    "games_match_medium": 0,
                    "match_medium_wins": 0,
                    "match_medium_wins_with_attempts": 0,
                    "match_medium_losses": 0,
                    "games_match_hard": 0,
                    "match_hard_wins": 0,
                    "match_hard_wins_with_attempts": 0,
                    "match_hard_losses": 0,
                }, file)

    def load_stats(self):
        with open("stats.json", "r") as file:
            self.stats = json.load(file)

    def save_stats(self):
        with open("stats.json", "w") as file:
            json.dump(self.stats, file)

    def run(self):
        self.create_stats()
        self.load_stats()
        logger.info("Game started")
        while self.state:
            self.state = self.game_properties[self.state]()
            
    def menu(self):
        print("-" * 50)
        print("[Play]\n[View Stats]\n[Exit]")
        print("-" * 50)
        while True:
            menu_input = input().lower()
            if menu_input in {"play"}:
                return "mode"
            elif menu_input in {"view stats"}:
                return "stats"
            elif menu_input in {"exit"}:
                print("-" * 50)
                print("Thx for playing!")
                print("-" * 50)
                logger.info("Player exited the game")
                logging.shutdown()
                sys.exit()
            else:
                print("-" * 50)
                print("Invalid command!")
                print("-" * 50)

    def view_stats(self):
        total_winRate = None
        guess_winRate = None
        match_winRate = None
        if (self.stats["games_played"] == 0):
            total_winRate = 0
        elif (self.stats["games_played"] > 0):
            total_winRate = round((self.stats["wins"] / self.stats["games_played"]) * 100, 2)
        if (self.stats["games_played_mode_guess"] == 0):
            guess_winRate = 0
        elif (self.stats["games_played_mode_guess"] > 0):
            guess_winRate = round((self.stats["wins_mode_guess"] / self.stats["games_played_mode_guess"]) * 100, 2)
        if (self.stats["games_played_mode_match"] == 0):  
            match_winRate = 0
        elif (self.stats["games_played_mode_match"] > 0):
            match_winRate = round((self.stats["wins_mode_match"] / self.stats["games_played_mode_match"]) * 100, 2)
        all_stats = [f"Games: {self.stats['games_played']}",
                    "-" * 50,
                    f"Wins: {self.stats['wins']}",
                    "-" * 50,
                    f"Wins with attempts left: {self.stats['wins_with_attempts']}",
                    "-" * 50,
                    f"Losses: {self.stats['losses']}",
                    "-" * 50,
                    f"Win rate: {total_winRate}%",
                    "-" * 50,
                    f"Games played on mode [Guess]: {self.stats['games_played_mode_guess']}",
                    "-" * 50, 
                    f"Wins on mode [Guess]: {self.stats['wins_mode_guess']}",
                    "-" * 50, 
                    f"Losses on mode [Guess]: {self.stats['losses_mode_guess']}",
                    "-" * 50,
                    f"Games played on mode [Guess] per difficulty: ",
                    "-" * 50,
                    f"Easy: {self.stats['games_guess_easy']}",
                    "-" * 50,
                    f"Medium: {self.stats['games_guess_medium']}",
                    "-" * 50,
                    f"Hard: {self.stats['games_guess_hard']}",
                    "-" * 50,
                    f"Wins on mode [Guess] per difficulty: ",
                    "-" * 50,
                    f"Easy: {self.stats['guess_easy_wins']}",
                    "-" * 50,
                    f"Medium: {self.stats['guess_medium_wins']}",
                    "-" * 50,
                    f"Hard: {self.stats['guess_hard_wins']}",
                    "-" * 50,
                    f"Wins with attempts on mode [Guess] per difficulty: ",
                    "-" * 50,
                    f"Medium: {self.stats['guess_medium_wins_with_attempts']}",
                    "-" * 50,
                    f"Hard: {self.stats['guess_hard_wins_with_attempts']}",
                    "-" * 50,
                    f"Losses on mode [Guess] per difficulty: ",
                    "-" * 50,
                    f"Medium: {self.stats['guess_medium_losses']}",
                    "-" * 50,
                    f"Hard: {self.stats['guess_hard_losses']}",
                    "-" * 50,
                    f"Win rate on mode [Guess]: {guess_winRate}%",
                    "-" * 50,
                    f"Games played on mode [Match]: {self.stats['games_played_mode_match']}",
                    "-" * 50, 
                    f"Wins on mode [Match]: {self.stats['wins_mode_match']}",
                    "-" * 50, 
                    f"Losses on mode [Match]: {self.stats['losses_mode_match']}",
                    "-" * 50,
                    f"Games played on mode [Match] per difficulty: ",
                    "-" * 50,
                    f"Easy: {self.stats['games_match_easy']}",
                    "-" * 50,
                    f"Medium: {self.stats['games_match_medium']}",
                    "-" * 50,
                    f"Hard: {self.stats['games_match_hard']}",
                    "-" * 50,
                    f"Wins on mode [Match] per difficulty: ",
                    "-" * 50,
                    f"Easy: {self.stats['match_easy_wins']}",
                    "-" * 50,
                    f"Medium: {self.stats['match_medium_wins']}",
                    "-" * 50,
                    f"Hard: {self.stats['match_hard_wins']}",
                    "-" * 50,
                    f"Wins with attempts on mode [Match] per difficulty: ",
                    "-" * 50,
                    f"Medium: {self.stats['match_medium_wins_with_attempts']}",
                    "-" * 50,
                    f"Hard: {self.stats['match_hard_wins_with_attempts']}",
                    "-" * 50,
                    f"Losses on mode [Match] per difficulty: ",
                    "-" * 50,
                    f"Medium: {self.stats['match_medium_losses']}",
                    "-" * 50,
                    f"Hard: {self.stats['match_hard_losses']}",
                    "-" * 50,
                    f"Win rate on mode [Match]: {match_winRate}%"]
        print("-" * 50)
        print("\n".join(all_stats))
        print("[Back]")
        print("-" * 50)
        while True:
            back_input = input().lower()
            if back_input in {"back"}:
                return "menu"
            else:
                print("-" * 50)
                print("Invalid command!")
                print("-" * 50)

    def mode_pick(self):
        print("-" * 50)
        print("[Guess]\n[Match]\n[Return]")
        print("-" * 50)
        while True:
            mode_input = input().lower()
            if mode_input in {"guess"}:
                logger.info("Mode selected: Guess")
                return "guess"
            elif mode_input in {"match"}:
                logger.info("Mode selected: Match")
                return "match"
            elif mode_input in {"return"}:
                logger.info("Player returned to menu")
                return "menu"
            else:
                print("Invalid command!")
    
    def guess_difficulty(self):
        print("-" * 50)
        print("[Easy]\n[Medium]\n[Hard]\n[Return]")
        print("-" * 50)
        while True:
            difficulty_input = input().lower()
            if difficulty_input in {"easy"}:
                self.last_guessDiff = "guess_easy"
                logger.info("Difficulty selected: easy")
                return "guess_easy"
            elif difficulty_input in {"medium"}:
                self.last_guessDiff = "guess_medium"
                logger.info("Difficulty selected: medium")
                return "guess_medium"
            elif difficulty_input in {"hard"}:
                self.last_guessDiff = "guess_hard"
                logger.info("Difficulty selected: hard")
                return "guess_hard"
            elif difficulty_input in {"return"}:
                logger.info("Player returned to menu")
                return "mode"
            else:
                print("Invalid choice!")

    def match_difficulty(self):
        print("-" * 50)
        print("[Easy]\n[Medium]\n[Hard]\n[Return]")
        print("-" * 50)
        while True:
            difficulty_input = input().lower()
            if difficulty_input in {"easy"}:
                self.last_matchDiff = "match_easy"
                logger.info("Difficulty selected: easy")
                return "match_easy"
            elif difficulty_input in {"medium"}:
                self.last_matchDiff = "match_medium"
                logger.info("Difficulty selected: medium")
                return "match_medium"
            elif difficulty_input in {"hard"}:
                self.last_matchDiff = "match_hard"
                logger.info("Difficulty selected: hard")
                return "match_hard"
            elif difficulty_input in {"return"}:
                logger.info("Returned to menu")
                return "mode"
            else:
                print("Invalid choice!")
                
    def game_prompts(self, random_number, guess):
        difference = abs(random_number - guess)
        if (25 >= difference > 15):
            print("Cold!")
        elif (100 >= difference > 25):
            print("Super cold!")
        elif (difference > 100):
            print("Extremely cold!")
        elif (15 >= difference > 10):
            print("Warm!")
        elif (self.last_guessDiff != "guess_easy" and 10 >= difference > 5):
            print("Very warm!")
        elif (self.last_guessDiff != "guess_easy" and 5 >= difference > 0):
            print("Really hot!")

    def stat_check(self, mode, diff, attempts, result):
        self.stats["games_played"] += 1
        if (mode == "guess"):
            self.stats["games_played_mode_guess"] += 1
            if (diff == "guess_easy" and result == "win"):
                self.stats["games_guess_easy"] += 1
                self.stats["wins"] += 1
                self.stats["wins_mode_guess"] += 1
                self.stats["guess_easy_wins"] += 1
            elif (diff == "guess_medium" and result == "win"):
                self.stats["games_guess_medium"] += 1
                self.stats["wins_mode_guess"] += 1
                self.stats["wins"] += 1
                self.stats["guess_medium_wins"] += 1
                if (attempts > 0 and attempts != math.inf):
                    self.stats["wins_with_attempts"] += 1
                    self.stats["guess_medium_wins_with_attempts"] += 1
            elif (diff == "guess_hard" and result == "win"):
                self.stats["games_guess_hard"] += 1
                self.stats["wins_mode_guess"] += 1
                self.stats["wins"] += 1
                self.stats["guess_hard_wins"] += 1
                if (attempts > 0 and attempts != math.inf):
                    self.stats["wins_with_attempts"] += 1
                    self.stats["guess_hard_wins_with_attempts"] += 1
    
            if (diff == "guess_medium" and result == "loss"):
                self.stats["games_guess_medium"] += 1
                self.stats["losses"] += 1
                self.stats["losses_mode_guess"] += 1
                self.stats["guess_medium_losses"] += 1
            elif (diff == "guess_hard" and result == "loss"):
                self.stats["games_guess_hard"] += 1 
                self.stats["losses"] += 1
                self.stats["losses_mode_guess"] += 1
                self.stats["guess_hard_losses"] += 1

        elif (mode == "match"):
            self.stats["games_played_mode_match"] += 1
            if (diff == "match_easy" and result == "win"):
                self.stats["games_match_easy"] += 1
                self.stats["wins"] += 1
                self.stats["wins_mode_match"] += 1
                self.stats["match_easy_wins"] += 1
            elif (diff == "match_medium" and result == "win"):
                self.stats["games_match_medium"] += 1
                self.stats["wins"] += 1
                self.stats["wins_mode_match"] += 1
                self.stats["match_medium_wins"] += 1
                if (attempts > 0 and attempts != math.inf):
                    self.stats["wins_with_attempts"] += 1
                    self.stats["match_medium_wins_with_attempts"] += 1
            elif (diff == "match_hard" and result == "win"):
                self.stats["games_match_hard"] += 1 
                self.stats["wins"] += 1
                self.stats["wins_mode_match"] += 1
                self.stats["match_hard_wins"] += 1
                if (attempts > 0 and attempts != math.inf):
                    self.stats["wins_with_attempts"] += 1
                    self.stats["match_hard_wins_with_attempts"] += 1
    
            if (diff == "match_medium" and result == "loss"):
                self.stats["games_match_medium"] += 1
                self.stats["losses"] += 1
                self.stats["losses_mode_match"] += 1
                self.stats["match_medium_losses"] += 1
            elif (diff == "match_hard" and result == "loss"):
                self.stats["games_match_hard"] += 1 
                self.stats["losses"] += 1
                self.stats["losses_mode_match"] += 1
                self.stats["match_hard_losses"] += 1

    def match_board(self, board, row_map):
        print("-" * 50)
        for i in range(1, 5):
            print("|".join(str(v) for v in board[row_map[i]].values()))
        print("-" * 50)
            
    def guessing_play(self, min_range, max_range, attempts_limit):
        random_number = random.randint(min_range, max_range)
        print("-" * 50)
        print("Guess!")
        while True:
            try:
                guess = int(input())
                    
                if (guess > random_number):
                    print("-" * 50)
                    print ("Too high!")
                    self.game_prompts(random_number, guess)
                    print("-" * 50)
                    attempts_limit -= 1
                elif (guess < random_number):
                    print("-" * 50)
                    print ("Too low!")
                    self.game_prompts(random_number, guess)
                    print("-" * 50)
                    attempts_limit -= 1
                elif (guess == random_number):
                    self.stat_check("guess", self.last_guessDiff, attempts_limit, "win")
                    self.save_stats()
                    logger.info(
                        f"Win | difficulty = {self.last_guessDiff} | number = {random_number}"
                    )
                    print("-" * 50)
                    print(f"Great guess!\nTry again?\nYes/No")
                    print("-" * 50)
                    self.mode = "guessed"
                    return "continue"
                        
                if (attempts_limit != math.inf and attempts_limit > 0):
                    print(f"Attempts left: {attempts_limit}")
                    print("-" * 50)
                    continue
                elif (attempts_limit == 0):
                    self.stat_check("guess" ,self.last_guessDiff, attempts_limit, "loss")
                    self.save_stats()
                    logger.info(
                        f"Loss | difficulty = {self.last_guessDiff} | number = {random_number}"
                    )
                    print(f"No attempts left!\nThe number was {random_number}\nTry again?\nYes/No")
                    print("-" * 50)
                    self.mode = "guessed"
                    return "continue"
                        
            except ValueError:
                print("-" * 50)
                print("Invalid guess!")
                print("-" * 50)

    def matching_play(self, min_range, max_range, attempts_limit): 
        matching_rows = {
            "first_row": {
                1: 0, 2: 0, 3: 0, 4: 0
            },
            "second_row": {
                1: 0, 2: 0, 3: 0, 4: 0
            },
            "third_row": {
                1: 0, 2: 0, 3: 0, 4: 0
            },
            "fourth_row": {
                1: 0, 2: 0, 3: 0, 4: 0
            }
        }
        matched_numbers = {
            "first_row": {
                1: "XXX", 2: "XXX", 3: "XXX", 4: "XXX"
            },
            "second_row": {
                1: "XXX", 2: "XXX", 3: "XXX", 4: "XXX"
            },
            "third_row": {
                1: "XXX", 2: "XXX", 3: "XXX", 4: "XXX"
            },
            "fourth_row": {
                1: "XXX", 2: "XXX", 3: "XXX", 4: "XXX"
            }
        }
        row_map = {
            1: "first_row",
            2: "second_row",
            3: "third_row",
            4: "fourth_row"
        }
        self.match_board(matched_numbers, row_map)
        number_generator = random.randint(min_range, max_range)
        exclude_generator = []
        exclude_generator.append(number_generator)
        counter = 1
        while counter < 8:
            number_generator = random.randint(min_range, max_range)
            if number_generator not in exclude_generator:
                exclude_generator.append(number_generator)
                counter += 1
        for number in exclude_generator:
            placed = 0
            while placed < 2:
                row = random.randint(1,4)
                col = random.randint(1,4)
                if matching_rows[row_map[row]][col] == 0:
                    matching_rows[row_map[row]][col] = number
                    placed += 1
        while matched_numbers != matching_rows:
            try:
                match_input = int(input())
                found =  False
                for row_key, row_value in matching_rows.items():
                    for col_key, col_value in matching_rows[row_key].items():
                        placed = 0
                        if (match_input == col_value):
                            while placed < 2:
                                matched_numbers[row_key][col_key] = col_value
                                found = True
                                placed += 1

                self.match_board(matched_numbers, row_map)
                
                if not found:
                    attempts_limit -= 1
                elif match_input in matching_rows:
                    logger.info("Number pair matched")
                    
                if (attempts_limit != math.inf and attempts_limit > 0):
                    print(f"Attempts left: {attempts_limit}")
                    print("-" * 50)
                    
                if (matched_numbers == matching_rows):
                    self.stat_check("match", self.last_matchDiff, attempts_limit, "win")
                    self.save_stats()
                    logger.info(f"Win | difficulty = {self.last_matchDiff}")
                    print("-" * 50)
                    print(f"Great job!\nTry again?\nYes/No")
                    print("-" * 50)
                    self.mode = "matched"
                    return "continue"
                elif (attempts_limit == 0):
                    self.stat_check("match", self.last_matchDiff, attempts_limit, "loss")
                    self.save_stats()
                    logger.info(f"Loss | difficulty = {self.last_matchDiff}")
                    print("No attempts left!\nTry again?\nYes/No")
                    self.mode = "matched"
                    return "continue"
            except ValueError:
                print("-" * 50)
                print("Invalid number!")
                print("-" * 50)

    def game_end(self):
        while True:
            choice = input().lower()
            if (self.mode == "guessed"):
                if (choice == "yes"):
                    logger.info("Player started a new game on the same difficulty")
                    return self.last_guessDiff
                elif (choice == "no"):
                    logger.info("Player ended the game")
                    return "menu"
                else:
                    print("-" * 50)
                    print("Invalid command!")
                    print("Try again?\nYes/No")
                    print("-" * 50)
                    
            elif (self.mode == "matched"):
                if (choice == "yes"):
                    logger.info("Player started a new game on the same difficulty")
                    return self.last_matchDiff
                elif (choice == "no"):
                    logger.info("Player ended the game")
                    return "menu"
                else:
                    print("-" * 50)
                    print("Invalid command!")
                    print("Try again?\nYes/No")
                    print("-" * 50)
                    
game = Game()
game.run()

--------------------------------------------------
[Play]
[View Stats]
[Exit]
--------------------------------------------------


 play


--------------------------------------------------
[Guess]
[Match]
[Return]
--------------------------------------------------


 match


--------------------------------------------------
[Easy]
[Medium]
[Hard]
[Return]
--------------------------------------------------


...

 back


--------------------------------------------------
[Play]
[View Stats]
[Exit]
--------------------------------------------------


 exit


--------------------------------------------------
Thx for playing!
--------------------------------------------------


SystemExit: 

In [ ]:
'''
finished the code for the new mode so that it counts attempts also
polished the code for the new mode
added stats for the new mode
added stats for the old mode so that there are stats for both modes as well as a few overall stats
polished printing the board
'''
#try to polish the code